# Gemma Model Analysis - Geometric Representation of Categorical Hierarchies

This notebook analyzes how semantic structure is encoded in the geometry of representation space for the Gemma-2b model.
We examine categorical and hierarchical concepts using the animals dataset.

In [ ]:
import torch
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
from tqdm import tqdm
import hierarchical as hrc
import pickle

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Gemma Model and Extract Representations

In [ ]:
# Load Gemma model
MODEL_NAME = "google/gemma-2b"
print(f"Loading model: {MODEL_NAME}")

# Get whitened unembeddings (g) and vocabulary
g, inv_sqrt_Cov_gamma, sqrt_Cov_gamma = hrc.get_g(MODEL_NAME, device)
vocab_dict, vocab_list = hrc.get_vocab(MODEL_NAME)

print(f"Vocabulary size: {len(vocab_list)}")
print(f"Embedding dimension: {g.shape[1]}")

## 2. Load Animals Hierarchical Data

In [ ]:
# Load animals hierarchical data
with open('data/animals.json', 'r') as f:
    animals_data = json.load(f)

# Define hierarchical structure
# Level 0: Animal (superclass)
# Level 1: mammal, bird, reptile, fish, amphibian, insect
categories = list(animals_data.keys())
print(f"Categories: {categories}")
print(f"\nSample words per category:")
for cat in categories:
    print(f"  {cat}: {len(animals_data[cat])} words (e.g., {animals_data[cat][:5]})") 

## 3. Map Words to Token Embeddings

In [ ]:
# Get animal category token embeddings
animals_token, animals_ind, animals_g = hrc.get_animal_category(
    animals_data, categories, vocab_dict, g
)

print("\nTokens found per category:")
for cat in categories:
    print(f"  {cat}: {len(animals_token[cat])} tokens")

## 4. Estimate Concept Vectors for Each Category

For each category, we estimate:
- LDA direction: optimal linear discriminant
- Mean vector: simple average of all word embeddings in the category

In [ ]:
# Estimate concept vectors for each category
concept_vectors = {}

for category in tqdm(categories, desc="Computing concept vectors"):
    lemmas = animals_token[category]
    if len(lemmas) > 0:
        concept_vectors[category] = hrc.estimate_cat_dir(lemmas, g, vocab_dict)
    else:
        print(f"Warning: No tokens found for category {category}")

print(f"\nComputed concept vectors for {len(concept_vectors)} categories")

## 5. Compute Projections onto Concept Vectors

For each category, compute projections of all words in that category onto the concept vector.
According to Theorem 4 of the paper, for a perfect concept, these projections should be constant.
In reality, they form a distribution - we measure the variance of this distribution.

In [ ]:
# Compute projections and statistics
projection_stats = {}

for category in categories:
    if category not in concept_vectors:
        continue
    
    # Get concept vector (LDA direction)
    lda_dir = concept_vectors[category]['lda']
    lda_dir_normalized = lda_dir / lda_dir.norm()
    
    # Get embeddings for words in this category
    category_embeddings = animals_g[category]
    
    # Compute projections
    projections = category_embeddings @ lda_dir_normalized
    projections_np = projections.cpu().numpy()
    
    # Store statistics
    projection_stats[category] = {
        'projections': projections_np,
        'mean': np.mean(projections_np),
        'std': np.std(projections_np),
        'variance': np.var(projections_np),
        'min': np.min(projections_np),
        'max': np.max(projections_np),
        'concept_vector_lda': lda_dir.cpu().numpy(),
        'concept_vector_mean': concept_vectors[category]['mean'].cpu().numpy()
    }

print("\nProjection Statistics (Intra-Class Variance):")
print(f"{'Category':<15} {'Mean':<10} {'Std':<10} {'Variance':<10}")
print("-" * 50)
for cat in categories:
    if cat in projection_stats:
        stats = projection_stats[cat]
        print(f"{cat:<15} {stats['mean']:<10.4f} {stats['std']:<10.4f} {stats['variance']:<10.6f}")

## 6. Visualize Projection Distributions

In [ ]:
# Plot histogram of projections for each category
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, category in enumerate(categories):
    if category in projection_stats:
        projections = projection_stats[category]['projections']
        axes[idx].hist(projections, bins=30, alpha=0.7, color='blue', edgecolor='black')
        axes[idx].axvline(projection_stats[category]['mean'], 
                         color='red', linestyle='--', linewidth=2, label='Mean')
        axes[idx].set_title(f"{category}\nσ={projection_stats[category]['std']:.4f}")
        axes[idx].set_xlabel('Projection Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend()

plt.tight_layout()
plt.savefig('gemma_projection_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nProjection distributions saved to 'gemma_projection_distributions.png'")

## 7. Compute Hierarchical Orthogonality

For hierarchical relationships (e.g., Animal → Mammal), we compute:
1. Parent vector (l_parent)
2. Child vector (l_child) 
3. Difference vector: d = l_child - l_parent
4. Cosine similarity between l_parent and d (should be ~0 for perfect orthogonality)

In [ ]:
# Create "Animal" parent concept from all animals
all_animal_tokens = []
for cat in categories:
    all_animal_tokens.extend(animals_token[cat])

# Remove duplicates
all_animal_tokens = list(set(all_animal_tokens))
print(f"Total unique animal tokens: {len(all_animal_tokens)}")

# Estimate parent concept vector for "Animal"
animal_concept = hrc.estimate_cat_dir(all_animal_tokens, g, vocab_dict)
l_parent = animal_concept['lda']
l_parent_normalized = l_parent / l_parent.norm()

print("\nComputed parent concept vector for 'Animal'")

In [ ]:
# Compute hierarchical orthogonality for each child category
hierarchical_stats = {}

for category in categories:
    if category not in concept_vectors:
        continue
    
    # Get child concept vector
    l_child = concept_vectors[category]['lda']
    l_child_normalized = l_child / l_child.norm()
    
    # Compute difference vector
    d = l_child - l_parent
    d_normalized = d / d.norm()
    
    # Compute cosine similarity between parent and difference
    cos_sim = (l_parent_normalized @ d_normalized).item()
    
    # Also compute direct cosine between parent and child
    cos_parent_child = (l_parent_normalized @ l_child_normalized).item()
    
    hierarchical_stats[category] = {
        'cos_parent_diff': cos_sim,
        'cos_parent_child': cos_parent_child,
        'l_child': l_child.cpu().numpy(),
        'l_parent': l_parent.cpu().numpy(),
        'd': d.cpu().numpy()
    }

print("\nHierarchical Orthogonality (Animal → Subcategory):")
print(f"{'Category':<15} {'cos(l_parent, d)':<20} {'cos(l_parent, l_child)':<20}")
print("-" * 60)
for cat in categories:
    if cat in hierarchical_stats:
        stats = hierarchical_stats[cat]
        print(f"{cat:<15} {stats['cos_parent_diff']:<20.6f} {stats['cos_parent_child']:<20.6f}")

print("\nNote: For perfect hierarchical orthogonality, cos(l_parent, d) should be close to 0")

## 8. Visualize Hierarchical Orthogonality

In [ ]:
# Plot hierarchical orthogonality
categories_list = list(hierarchical_stats.keys())
cos_parent_diff = [hierarchical_stats[cat]['cos_parent_diff'] for cat in categories_list]
cos_parent_child = [hierarchical_stats[cat]['cos_parent_child'] for cat in categories_list]

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
x = np.arange(len(categories_list))
width = 0.35

ax.bar(x - width/2, cos_parent_diff, width, label='cos(l_parent, d)', alpha=0.8)
ax.bar(x + width/2, cos_parent_child, width, label='cos(l_parent, l_child)', alpha=0.8)
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Category')
ax.set_ylabel('Cosine Similarity')
ax.set_title('Hierarchical Orthogonality: Animal → Subcategories (Gemma)')
ax.set_xticks(x)
ax.set_xticklabels(categories_list, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('gemma_hierarchical_orthogonality.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nHierarchical orthogonality plot saved to 'gemma_hierarchical_orthogonality.png'")

## 9. Save Results for Comparison

In [ ]:
# Save all results for later comparison with Vault Gemma
gemma_results = {
    'model_name': MODEL_NAME,
    'projection_stats': projection_stats,
    'hierarchical_stats': hierarchical_stats,
    'concept_vectors': {cat: {'lda': concept_vectors[cat]['lda'].cpu().numpy(),
                               'mean': concept_vectors[cat]['mean'].cpu().numpy()}
                       for cat in concept_vectors},
    'categories': categories,
    'animals_token': animals_token,
    'animals_ind': animals_ind
}

with open('gemma_results.pkl', 'wb') as f:
    pickle.dump(gemma_results, f)

print("\nResults saved to 'gemma_results.pkl'")
print("\n=== Gemma Analysis Complete ===")

## Summary

This notebook analyzed the Gemma-2b model's geometric representation of categorical hierarchies:

1. **Intra-Class Variance**: Measured the variance of projections onto concept vectors for each category
2. **Hierarchical Orthogonality**: Computed the orthogonality between parent concepts and child-specific directions

Results have been saved for comparison with Vault Gemma in the evaluation notebook.